Verify if boxes are drawn accurately

In [ ]:
from collections import defaultdict
import os
import json
import re
from typing import List, Tuple
import cv2
from glob import glob
import numpy as np
from datetime import datetime, time
from dotenv import load_dotenv
from google.genai import types
from google.genai.errors import ServerError
from pathlib import Path
from utils import load_service_account
from prompts import img_ground_truth_prompt
from schemas import test_llm_schema
from dataset_stats import stats_process
from test_llm import process_llm_images, process_llm_map
from verify_gt import image_test_process, scale_boxes, scale_non_square, draw_gtboxes
from resize_dataset import resize_images_process

Variables

In [ ]:
gt_dir = "resources/ground_truth"
DATASET_FOLDER = "datasets/neo_weapons" # "create dataset/dangerous_weapons/images"
TEST_IMAGE_PATH = f"{gt_dir}/000000000090.jpg"
STATS_JSON = "output/stats/neo_image_stats.json"
target_size = 320 # Dataset square image size

OUTPUT_DIR = f"{gt_dir}/gt_images"
gt_ver_dir = f"{gt_dir}/gt_verify"
llm_dir = f"{gt_dir}/llm_images"
os.makedirs(gt_dir, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(gt_ver_dir, exist_ok=True)
os.makedirs(llm_dir, exist_ok=True)

file_title="llm_verify_box"
dataset_title="verify_box"
master_map_path = os.path.join(gt_dir, "llm_scaling_map.json")

In [ ]:
stats_process(DATASET_FOLDER, STATS_JSON)

Add Ground Truth Box Coordinates if known. Can also use `app.py` to get coordinates (x1,y1,x2,y2). You will have to manually switch the x and y positions when inputting into the GT_BOXES.

In [ ]:
# Ground truth boxes (pixel coords)
# Example: [[ymin, xmin, ymax, xmax], ...]
GT_BOXES = [
    [0, 0, 0, 0]
]
LABEL_CLASSES = {"cow"}
color = (0, 255, 0)  # Green for GT boxes
norm_range=1000
pixel_dims=False # False = Normalize to 0-norm_range, True = Keep in pixel dimensions
trials = 3 # number of runs to average over, set to 1 for no averaging (just one run)

Continue

In [ ]:
output_path = image_test_process(STATS_JSON, TEST_IMAGE_PATH, GT_BOXES, OUTPUT_DIR)
if GT_BOXES == [[0, 0, 0, 0]]:
    print("No GT boxes provided. Skipping GT box drawing.")
    gt_ver_dir = OUTPUT_DIR  # Use the same directory for verification if no GT boxes
    dataset_title="" # Clear dataset title since there are no GT boxes to verify against
else:
    draw_gtboxes(output_path, OUTPUT_DIR, color, gt_ver_dir)

Continue

In [ ]:
for trial in range(1, trials + 1):
    folder_path= f"{gt_dir}/llm_images test {trial}"
    os.makedirs(folder_path, exist_ok=True)
    process_llm_images(gt_ver_dir, folder_path, norm_range=norm_range, LABEL_CLASSES=LABEL_CLASSES)

Continue

In [ ]:
map_path = process_llm_map(master_map_path, pixel_dims=pixel_dims, norm_range=norm_range, tolerance=0.02)
draw_gtboxes(master_map_path, gt_ver_dir, (0, 0, 255), llm_dir, file_title, dataset_title)